# NSE Trading Lab — Getting Started

This notebook walks you through the basics of backtesting strategies on NSE stocks.

In [ ]:
import sys
sys.path.insert(0, '..')

from nse_backtest import (
    fetch_nse, fetch_nifty50, NIFTY50_SYMBOLS,
    STRATEGIES, run_backtest, TradeConfig,
    compute_metrics, print_report, plot_results, compare_strategies
)

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Fetch Stock Data

In [ ]:
# Fetch any NSE stock
data = fetch_nse('RELIANCE', start='2018-01-01')
data.tail()

## 2. Run a Single Strategy

In [ ]:
# Apply Supertrend strategy
from nse_backtest.strategies import supertrend

strat_data = supertrend(data, period=10, multiplier=3.0)

# Configure with your capital and risk
config = TradeConfig(
    initial_capital=100_000,  # 1 lakh
    stop_loss_pct=0.07,       # 7% stop loss
)

result = run_backtest(strat_data, config)
metrics = compute_metrics(result)
print_report(metrics, 'RELIANCE — Supertrend')

In [ ]:
# Visualize
plot_results(result, metrics, 'RELIANCE — Supertrend', save_path='../output/reliance_supertrend.png')

## 3. Compare All Strategies on One Stock

In [ ]:
all_results = []
for name, strat_func in STRATEGIES.items():
    try:
        sd = strat_func(data)
        res = run_backtest(sd, config)
        met = compute_metrics(res)
        label = sd['strategy_name'].iloc[-1]
        all_results.append((label, res, met))
        print(f"{label:<35} Sharpe={met['sharpe_ratio']:.2f}  CAGR={met['cagr_pct']:.1f}%")
    except Exception as e:
        print(f"Error in {name}: {e}")

compare_strategies(all_results, save_path='../output/reliance_comparison.png')

## 4. Scan Multiple Stocks

Find which Nifty 50 stocks respond best to your chosen strategy.

In [ ]:
# Scan top 10 Nifty stocks with Supertrend
scan_results = []
test_symbols = NIFTY50_SYMBOLS[:10]  # Start with 10, expand later

for sym in test_symbols:
    try:
        stock_data = fetch_nse(sym, start='2020-01-01')
        sd = supertrend(stock_data)
        res = run_backtest(sd, config)
        met = compute_metrics(res)
        scan_results.append({
            'symbol': sym,
            'sharpe': met['sharpe_ratio'],
            'cagr': met['cagr_pct'],
            'max_dd': met['max_drawdown_pct'],
            'win_rate': met['win_rate_pct'],
            'trades': met['total_trades'],
        })
    except Exception as e:
        print(f"Skip {sym}: {e}")

import pandas as pd
scan_df = pd.DataFrame(scan_results).sort_values('sharpe', ascending=False)
scan_df

## 5. Test Your ARSSBL Position

Let's see what the data says about your current holding.

In [ ]:
# Uncomment and run when you're ready
# arssbl = fetch_nse('ARSSBL', start='2022-01-01')
# for name, strat_func in STRATEGIES.items():
#     sd = strat_func(arssbl)
#     res = run_backtest(sd, config)
#     met = compute_metrics(res)
#     label = sd['strategy_name'].iloc[-1]
#     print(f"{label:<35} Sharpe={met['sharpe_ratio']:.2f}  Return={met['total_return_pct']:.1f}%")